# 09. Point processes inferred by SVI (BSTPP), and comparison with the SP mixture

This notebook fits three point processes to each dataset with the **BSTPP** library
(Manring et al. 2025, JAX + NumPyro), by stochastic variational inference (SVI). The
parameters are inferred **from the data**, without any heuristic. The fitted models are
the competitors of the SP mixture in notebook 07 and in `scripts/compute_all.py`.

1. **LGCP**, log-Gaussian Cox process, intensity λ(s) = exp(a₀ + f_s(s)) with f_s a
   Gaussian process.
2. **Hawkes**, self-exciting process, λ(t, s) = μ + Σ α f(t − tᵢ) φ(s − sᵢ).
3. **Cox-Hawkes**, a Gaussian-process background plus self-excitation.

**Limitation.** BSTPP models are spatio-temporal, whereas our data are purely spatial
(snapshots without time stamps). Artificial times T ~ U(0, T_max) are assigned, which
makes the temporal part a nuisance parameter. The spatial part of the inference (α, σ,
Gaussian-process field) stays meaningful.

**Cost and outputs.** The full inference takes about one hour (5 000 SVI steps per model,
eight datasets). Models that already exist in `out/bstpp_results/` are reloaded instead
of being fitted again, unless `FORCE_FIT = True`. New fits are saved in
`out/bstpp_results/`, except when this folder is a symbolic link to another repository,
in which case they go to `out/bstpp_reinferred/` so that the linked files are never
overwritten.

The SVI uses at most `BSTPP_CAP = 2000` points per dataset (memory grows as N² in the
likelihood). The models are then simulated, and their 2PCF is compared with the data and
with the SP mixture at the same number of nodes.

References: Manring et al. (2025), *BSTPP: a Bayesian spatiotemporal point process
package*, https://github.com/manninglab/BSTPP. Soneira & Peebles (1978), ApJ 211, 1.

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from shapely.geometry import box as _shapely_box

import jax
import numpyro
import numpyro.distributions as dist
from bstpp.main import LGCP_Model, Hawkes_Model

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))
import synthesis_utils as su

DATASET_NAMES = list(su.DATASETS)      # datasets whose files are missing are skipped
FORCE_FIT = False                      # True: fit again even if a .pkl exists
BSTPP_LR = 0.01
BSTPP_NUM_STEPS = 5000                 # 10 000+ for a more precise inference
N_SIM = 5                              # simulations per model for the 2PCF
SEED = su.SEED
T_MAX = 365.0                          # artificial time window (one year)
# Domain [0, 1]^2 as a GeoDataFrame. Passing a numpy array instead triggers a bug
# in BSTPP's _sim_cox (self-join of the computation grid, intensities x5, OOM).
A_GDF = gpd.GeoDataFrame(geometry=[_shapely_box(0.0, 0.0, 1.0, 1.0)])

LOAD_DIR = su.BSTPP_RESULTS
SAVE_DIR = (ROOT / "out" / "bstpp_reinferred") if LOAD_DIR.is_symlink() else LOAD_DIR
SAVE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = ROOT / "out" / "bstpp"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 11,
                     "axes.labelsize": 10, "font.size": 9})
print(f"JAX {jax.__version__} | NumPyro {numpyro.__version__}")
print(f"SVI steps={BSTPP_NUM_STEPS}  lr={BSTPP_LR}  cap={su.BSTPP_CAP}")
print(f"models read from {LOAD_DIR.relative_to(ROOT)}, new fits saved to "
      f"{SAVE_DIR.relative_to(ROOT)}")

## Data

`su.load_dataset` reads the border and the points, normalises the coordinates to
[0, 1]² and computes the data 2PCF. The SVI training sample is rebuilt by
`su._bstpp_xy` (at most 2 000 points, seed 123), the same one used when the models are
reloaded by `su.load_bstpp_models`.

In [ ]:
loaded = {}
for name in DATASET_NAMES:
    ds = su.DATASETS[name]
    if not (Path(ds["path_border"]).exists() and Path(ds["path_csv"]).exists()):
        print(f"[{name}] skipped, data files not found")
        continue
    print(f"\n--- {name}")
    loaded[name] = su.load_dataset(ds)

## Inference

Protocol: normalised coordinates with artificial times, SVI with `BSTPP_NUM_STEPS`
steps, posterior samples in `model.samples`. The temporal parameters (β, f_t) are
nuisance parameters.

In [ ]:
def training_frame(ld, seed=42):
    """BSTPP data frame of the training sample, sorted by the artificial time T."""
    xy = su._bstpp_xy(ld)
    T = np.random.default_rng(seed).uniform(0, T_MAX, len(xy))
    return (pd.DataFrame({"X": xy[:, 0], "Y": xy[:, 1], "T": T})
            .sort_values("T").reset_index(drop=True))


def new_model(mname, df):
    """Untrained BSTPP model with the priors used for the paper."""
    if mname == "LGCP":
        return LGCP_Model(df, A_GDF, T_MAX, a_0=dist.Normal(0, 5))
    return Hawkes_Model(df, A_GDF, T_MAX, cox_background=(mname == "Cox Hawkes"),
                        a_0=dist.Normal(0, 5), alpha=dist.Beta(2, 5),
                        beta=dist.HalfNormal(2.0), sigmax_2=dist.HalfNormal(0.25))


def pkl_name(name, mname):
    return f"{name.replace(' ', '_')}_{mname.replace(' ', '_')}.pkl"


bstpp_models = {}
for name, ld in loaded.items():
    print(f"\n{'=' * 55}\n  {name}\n{'=' * 55}")
    df = training_frame(ld)
    bstpp_models[name] = {}
    for mname in su.BSTPP_NAMES:
        pkl_in = LOAD_DIR / pkl_name(name, mname)
        try:
            model = new_model(mname, df)
            if pkl_in.exists() and not FORCE_FIT:
                model.load_rslts(str(pkl_in))
                print(f"  [{mname}] reloaded from {pkl_in.name}")
            else:
                print(f"  [{mname}] SVI, {BSTPP_NUM_STEPS} steps ...")
                model.run_svi(lr=BSTPP_LR, num_steps=BSTPP_NUM_STEPS, plot_loss=False)
                model.save_rslts(str(SAVE_DIR / pkl_name(name, mname)))
                print(f"  [{mname}] fitted, saved to {SAVE_DIR.name}/")
            bstpp_models[name][mname] = model
        except Exception as e:
            print(f"  [{mname}] FAILED: {e}")
            bstpp_models[name][mname] = None

## Posterior summary

In [ ]:
post_rows = []
for name, models in bstpp_models.items():
    for mname, model in models.items():
        if model is None:
            continue
        for k in ("a_0", "alpha", "beta", "sigmax_2"):
            if k in model.samples:
                v = np.asarray(model.samples[k])
                if v.ndim == 1:
                    post_rows.append(dict(dataset=name, model=mname, parameter=k,
                                          mean=v.mean(), std=v.std(),
                                          q05=np.percentile(v, 5),
                                          q95=np.percentile(v, 95)))
posterior = pd.DataFrame(post_rows)
posterior.round(4)

## Simulation and 2PCF

Each model is simulated `N_SIM` times from its posterior. Each realisation is restricted
to the border and resampled to N = min(N_real, 2000) points, the size of the training
sample (asking for more would be out of distribution). The SP mixture and the data are
taken at the same N.

In [ ]:
def model_2pcf(model, geom_norm, N, n_sim=N_SIM, seed=SEED):
    """Mean 2PCF over n_sim realisations, and one realisation for the maps."""
    r_ref, xis, xy_ex = None, [], None
    for k in range(n_sim):
        xy = su._bstpp_realization(model, geom_norm, N, seed + k)
        if xy is None:
            continue
        r, xi = su.compute_2pcf_norm(xy, geom_norm, n_runs=3)
        if r_ref is None:
            r_ref, xy_ex = r, xy
        xis.append(su._align_xi(r, xi, r_ref))
    if r_ref is None:
        return None, None, None
    return r_ref, np.nanmean(xis, axis=0), xy_ex


results = {}
for name, ld in loaded.items():
    ds = su.DATASETS[name]
    N = min(ld["N_real"], su.BSTPP_CAP)
    print(f"\n--- {name}  (N = {N})")
    xy_d, r_d, xi_d = su.data_2pcf_at_N(ld, N)
    xy_sp = su.build_sp_mixture(ds, ld["gdf_edge"], ld["bounds"], ld["geom_norm"],
                                ld["r_data"], ld["xi_data"], N)
    r_sp, xi_sp = su.compute_2pcf_norm(xy_sp, ld["geom_norm"])
    res = {"data": (r_d, xi_d, xy_d), "SP": (r_sp, xi_sp, xy_sp)}
    for mname, model in bstpp_models[name].items():
        res[mname] = model_2pcf(model, ld["geom_norm"], N) if model is not None \
            else (None, None, None)
        print(f"  [{mname}] {'ok' if res[mname][0] is not None else 'no realisation'}")
    results[name] = res

## Maps and 2PCF

In [ ]:
ORDER = ["data", "SP", "LGCP", "Hawkes", "Cox Hawkes"]
COLORS = {"data": "black", "SP": "#2ca02c", "LGCP": "#1f77b4", "Hawkes": "#ff7f0e",
          "Cox Hawkes": "#d62728"}
STYLES = {"data": dict(lw=3.0, ls="-", zorder=10), "SP": dict(lw=2.5, ls="--", zorder=9),
          "LGCP": dict(lw=2.0, ls="-.", zorder=7),
          "Hawkes": dict(lw=2.0, ls=(0, (3, 1, 1, 1)), zorder=7),
          "Cox Hawkes": dict(lw=2.0, ls=":", zorder=8)}


def plot_boundary(ax, geom, color="#444444", lw=0.8):
    for g in (geom.geoms if hasattr(geom, "geoms") else [geom]):
        x, y = g.exterior.xy
        ax.plot(x, y, color=color, lw=lw, zorder=0)


rows = []
for name, res in results.items():
    ds, ld = su.DATASETS[name], loaded[name]
    r_d, xi_d, _ = res["data"]
    jsd = {m: (su.js_divergence(r_d, xi_d, res[m][0], res[m][1])
               if res[m][0] is not None else np.nan) for m in ORDER[1:]}

    fig = plt.figure(figsize=(20, 9))
    gs = gridspec.GridSpec(2, 5, figure=fig, hspace=0.35, wspace=0.25)
    for col, m in enumerate(ORDER):
        ax = fig.add_subplot(gs[0, col])
        xy = res[m][2]
        if xy is not None and len(xy):
            ax.scatter(xy[:, 0], xy[:, 1], s=max(0.5, 3.5 - len(xy) / 2500), alpha=0.35,
                       color=COLORS[m], rasterized=True)
        else:
            ax.text(0.5, 0.5, "no realisation", ha="center", va="center",
                    transform=ax.transAxes, color="gray")
        plot_boundary(ax, ld["geom_norm"])
        ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        sub = f"\nJSD={jsd[m]:.4f}" if m in jsd and np.isfinite(jsd[m]) else ""
        ax.set_title(m + sub, color=COLORS[m], fontweight="bold", fontsize=9)

    ax = fig.add_subplot(gs[1, :])
    for m in ORDER:
        r, xi, _ = res[m]
        if r is None:
            continue
        ok = np.isfinite(xi) & (r > 0)
        lab = m if m == "data" else f"{m}  (JSD={jsd[m]:.4f})"
        ax.plot(r[ok] * ld["scale_m"] / 1e3, xi[ok], color=COLORS[m], label=lab,
                **STYLES[m])
    ax.axhline(0, color="gray", lw=0.7, ls="--")
    ax.set_xscale("log"); ax.set_ylim(bottom=-0.5)
    ax.set_xlabel("r [km]"); ax.set_ylabel(r"$\hat{\xi}(r)$  (Landy-Szalay)")
    ax.set_title(f"2PCF, {name} (γ_data = {ds['gamma_data']:.3f})")
    ax.legend(fontsize=8); ax.grid(alpha=0.25)
    fig.suptitle(f"SP against BSTPP, {name} (N = {len(res['data'][2]):,})",
                 fontsize=13, fontweight="bold")
    fig.savefig(FIG_DIR / f"comparison_{name.replace(' ', '_')}_BSTPP.pdf",
                bbox_inches="tight", dpi=150)
    plt.show()

    for m in ORDER[1:]:
        r, xi, _ = res[m]
        g, _ = su.fit_powerlaw_norm(r, xi) if r is not None else (None, None)
        aic = np.nan
        if m in bstpp_models[name] and bstpp_models[name][m] is not None:
            try:
                aic = float(bstpp_models[name][m].expected_AIC())
            except Exception:
                pass
        rows.append(dict(dataset=name, model=m, gamma=g,
                         dgamma=(g - ds["gamma_data"]) if g is not None else np.nan,
                         jsd=jsd[m], aic=aic))

## Summary table, γ, JSD and AIC

In [ ]:
table = pd.DataFrame(rows)
table.to_csv(ROOT / "out" / "09_bstpp_comparison.csv", index=False)
print(table.round(4).to_string(index=False))
print("\nJSD rank per dataset (1 = best)")
for name, sub in table.dropna(subset=["jsd"]).groupby("dataset", sort=False):
    order = sub.sort_values("jsd")
    print(f"  {name}: " + ", ".join(f"{i}. {m} ({j:.4f})" for i, (m, j) in
                                   enumerate(zip(order.model, order.jsd), 1)))

## What BSTPP brings, and its limits

- **Inference from the data.** The parameters (α, σ, a₀, Gaussian-process field) are
  inferred by SVI, with a full posterior distribution, so the parameter uncertainty is
  quantified. There is no m-scan, no silhouette and no quadrat heuristic.
- **Model selection.** The expected AIC compares the models on their likelihood.
- **Limits.** The data are purely spatial, so the temporal parameters (β, f_t) cannot be
  interpreted. The Gaussian process of the LGCP and Cox-Hawkes models has a finite range
  (its VAE is pre-trained on a fixed Matérn kernel), which can prevent them from
  reproducing a power-law 2PCF over several decades.
- **Cost.** Several minutes of SVI per model and per dataset, against a fraction of a
  second for the SP calibration (notebook 08).